# Figure 2C and 2D: ICI vs Non-ICI, and CTLA-4 vs PD-(L)1

Python port of `Cumulative_Incidence_By_ICI_Strata_2E.R`, limited to the two
requested stratifications (Stratum A and Stratum B from the R script; Stratum
C — multi-class vs single-class — is intentionally omitted here).

- **Figure 2C** — ICI (any regimen containing a checkpoint inhibitor) vs Non-ICI.
- **Figure 2D** — within ICI patients only: CTLA-4 (± PD-(L)1) vs PD-(L)1 (no CTLA-4).
  Patients flagged as ICI only through the generic `contains_immuno` column,
  with no agent specified, are held out of Figure B (same as the R script).

Same cohort, censoring window, Kaplan–Meier estimator, and Cox modeling
approach as the R script and its source notebook.

## 1. Imports & global config

In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'main'))
os.makedirs(RESULTS_DIR, exist_ok=True)

%matplotlib inline

import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 450,
})
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

TOXICITY_COLUMNS = ["liver_toxicity", "hypothyroidism", "pneumonitis",
                     "colitis", "adrenal_insufficiency", "hyperthyroidism"]
TOXICITY_DISPLAY = {
    "pneumonitis": "Pneumonitis", "adrenal_insufficiency": "Adrenal Insufficiency",
    "liver_toxicity": "Liver Toxicity", "colitis": "Colitis",
    "hyperthyroidism": "Hyperthyroidism", "hypothyroidism": "Hypothyroidism",
}

T_MONTHS = 12.0
DAYS_PER_MONTH = 30.44
MIN_EVENTS_PER_PARAM = 10
PENALIZER = 0.1  # light L2, safety net for near-separation

_ICI_COLOR = "#3C5488"
_NON_ICI_COLOR = "#E64B35"
STRATUM_A_COLORS = {"ICI": _ICI_COLOR, "Non-ICI": _NON_ICI_COLOR}
STRATUM_B_COLORS = {"CTLA-4 (± PD-(L)1)": "#117A65", "PD-(L)1 (no CTLA-4)": "#3498DB"}

STRATUM_A_ORDER = ["ICI", "Non-ICI"]
STRATUM_B_ORDER = ["CTLA-4 (± PD-(L)1)", "PD-(L)1 (no CTLA-4)"]

## 2. Helper functions

In [ ]:
def standardize_mrn(mrn):
    """Extract the first digit run and zero-pad to 8 characters."""
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r"\d+", str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None


def on(row, col):
    """The notebook's `row.get(col, 0) in [1, True, "1", "True"]`."""
    return row.get(col, 0) in [1, True, "1", "True"]

## 3. Load & standardize data

In [ ]:
BACKBONE_PATH = os.path.join(TOX_TABLE_DIR, 'llm84k_pneumonitis_grade0_20260630.csv')
covars = pd.read_csv(BACKBONE_PATH, low_memory=False)
batch_df = pd.read_csv(os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv'), encoding="latin-1", low_memory=False)

covars["mrn"] = covars["mrn"].apply(standardize_mrn)
batch_df["mrn"] = batch_df["mrn"].apply(standardize_mrn)
covars = covars[covars["mrn"].notna()].copy()
batch_df = batch_df[batch_df["mrn"].notna()].copy()

covars["lot_start"] = pd.to_datetime(covars["lot_start"], errors="coerce")
covars["lot"] = pd.to_numeric(covars["lot"], errors="coerce")
covars["censor_days"] = pd.to_numeric(covars["t_cutoff_lot"], errors="coerce")
covars = covars.sort_values(["mrn", "lot"])

batch_df["window_start"] = pd.to_datetime(batch_df["window_start"], errors="coerce")
batch_df["window_end"] = pd.to_datetime(batch_df["window_end"], errors="coerce")
batch_df = batch_df.dropna(subset=["window_start", "window_end"])

print(f"Covars: {covars.shape}")
print(f"Batch: {batch_df.shape}, {batch_df['mrn'].nunique():,} patients")

## 4. One row per patient, with both grouping variables

In [ ]:
# One row per patient — first non-null value per column
cols_needed = ["mrn"] + [c for c in covars.columns if c.startswith("contains_")] + \
    ["age_at_lot_start", "sex", "cancer_type"]
cols_needed = [c for c in cols_needed if c in covars.columns]

patient_covars = (
    covars[cols_needed]
    .groupby("mrn", as_index=False)
    .agg(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)
)

def _row_flags(row):
    has_ctla4 = on(row, "contains_ctla4_immuno") or on(row, "contains_ctla4")
    has_pd1 = on(row, "contains_non_ctla4_immuno") or on(row, "contains_pd1")
    is_ici = on(row, "contains_immuno") or has_ctla4 or has_pd1
    return pd.Series({"has_ctla4": has_ctla4, "has_pd1": has_pd1, "is_ici": is_ici})

flags = patient_covars.apply(_row_flags, axis=1)
patient_covars = pd.concat([patient_covars, flags], axis=1)

# ---- Stratum A: ICI vs Non-ICI (whole cohort) --------------------------------
patient_covars["ici_group"] = np.where(patient_covars["is_ici"], "ICI", "Non-ICI")

# ---- Stratum B: CTLA-4 (\u00b1 PD-(L)1) vs PD-(L)1 (no CTLA-4), ICI patients only --
# CTLA-4-dominant by design: any CTLA-4 exposure lands in the CTLA-4 arm even
# when PD-(L)1 is also present.
def _agent(row):
    if row["has_ctla4"]:
        return "CTLA-4 (\u00b1 PD-(L)1)"
    if row["has_pd1"]:
        return "PD-(L)1 (no CTLA-4)"
    if row["is_ici"]:
        return "ICI, agent unspecified"  # held out of Stratum B
    return np.nan

patient_covars["ici_agent"] = patient_covars.apply(_agent, axis=1)

print(patient_covars["ici_group"].value_counts())
print()
print(patient_covars["ici_agent"].value_counts(dropna=False))

## 5. Line-1 window and censoring

Each patient is followed from their first line-of-therapy (LOT) start until
the earliest of: 180 days after LOT-1 ends, LOT-2 start, death, or last
follow-up.

In [ ]:
covars_valid = covars[covars["lot_start"].notna()].copy()
line1 = (covars_valid.sort_values(["mrn", "lot"])
         .groupby("mrn").first().reset_index()[["mrn", "lot_start", "censor_days"]]
         .rename(columns={"lot_start": "line1_start"}))
line1 = line1[np.isfinite(line1["censor_days"]) & (line1["censor_days"] > 0)].copy()
print(f"Line 1 patients with valid censoring: {len(line1):,}")

# Keep only AE calls that fall inside each patient's Line-1 window.
batch_merged = batch_df.merge(line1, on="mrn", how="inner")
batch_merged["days_from_start"] = (batch_merged["window_start"] - batch_merged["line1_start"]).dt.days
batch_merged = batch_merged[
    (batch_merged["days_from_start"] >= 0) &
    (batch_merged["days_from_start"] <= batch_merged["censor_days"])
].copy()

## 6. Build the two group -> MRN-set lookups

In [ ]:
all_mrns = set(line1["mrn"])

# Stratum A groups
ici_map = patient_covars.set_index("mrn")["ici_group"].to_dict()
stratum_a_mrns = {
    grp: {m for m in all_mrns if ici_map.get(m) == grp}
    for grp in STRATUM_A_ORDER
}

# Stratum B groups (ICI patients with a specified agent only)
agent_map = patient_covars.set_index("mrn")["ici_agent"].to_dict()
stratum_b_mrns = {
    grp: {m for m in all_mrns if agent_map.get(m) == grp}
    for grp in STRATUM_B_ORDER
}

print("Stratum A (N):")
for g in STRATUM_A_ORDER:
    print(f"  {g}: {len(stratum_a_mrns[g]):,}")

print("\nStratum B (N):")
for g in STRATUM_B_ORDER:
    print(f"  {g}: {len(stratum_b_mrns[g]):,}")

## 7. Cumulative-incidence function

Same KM logic as the source notebook: `lifelines`' default confidence
interval is the exponential-Greenwood ("log-log") interval, matching the R
script's `conf.type = "log-log"`.

In [ ]:
def ci_at_t(mrn_set, tox):
    sub_l1 = line1[line1["mrn"].isin(mrn_set)].copy()
    sub_b = batch_merged[batch_merged["mrn"].isin(mrn_set)].copy()
    if len(sub_l1) < 10:
        return 0.0, 0.0, 0.0

    ae_rec = sub_b[sub_b[tox] == 1].copy() if tox in sub_b.columns else pd.DataFrame()
    ae_rec = ae_rec[ae_rec["days_from_start"] >= 0] if len(ae_rec) > 0 else ae_rec

    if len(ae_rec) > 0:
        first_ae = (ae_rec.groupby("mrn")["days_from_start"].min()
                    .reset_index().rename(columns={"days_from_start": "time"}))
        first_ae["event"] = 1
    else:
        first_ae = pd.DataFrame(columns=["mrn", "time", "event"])

    surv = sub_l1[["mrn", "censor_days"]].merge(first_ae, on="mrn", how="left")
    surv["event"] = surv["event"].fillna(0).astype(int)
    surv.loc[surv["event"] == 0, "time"] = surv.loc[surv["event"] == 0, "censor_days"]
    surv.loc[(surv["event"] == 1) & (surv["time"] > surv["censor_days"]), "event"] = 0
    surv.loc[surv["event"] == 0, "time"] = surv["censor_days"]
    surv = surv[surv["time"] > 0].copy()
    if len(surv) < 10:
        return 0.0, 0.0, 0.0

    surv["time_months"] = surv["time"] / DAYS_PER_MONTH
    kmf = KaplanMeierFitter()
    kmf.fit(surv["time_months"], surv["event"])

    ci = (1 - kmf.survival_function_at_times(T_MONTHS).values[0]) * 100
    ci_tbl = kmf.confidence_interval_survival_function_
    idx = max(0, min(np.searchsorted(kmf.survival_function_.index, T_MONTHS, side="right") - 1,
                      len(ci_tbl) - 1))
    lo = (1 - ci_tbl.iloc[idx, 1]) * 100
    hi = (1 - ci_tbl.iloc[idx, 0]) * 100
    return ci, lo, hi


def build_surv_table(mrn_set, tox):
    """Per-patient (time_months, event) table for a group/toxicity — same
    censoring logic as ci_at_t, but returns the raw table instead of a KM
    summary. Used for both the Cox loop and the log-rank p-values below."""
    sub_l1 = line1[line1["mrn"].isin(mrn_set)].copy()
    sub_b = batch_merged[batch_merged["mrn"].isin(mrn_set)].copy()

    ae_rec = sub_b[sub_b[tox] == 1].copy() if tox in sub_b.columns else pd.DataFrame()
    ae_rec = ae_rec[ae_rec["days_from_start"] >= 0] if len(ae_rec) > 0 else ae_rec

    if len(ae_rec) > 0:
        first_ae = (ae_rec.groupby("mrn")["days_from_start"].min()
                    .reset_index().rename(columns={"days_from_start": "time"}))
        first_ae["event"] = 1
    else:
        first_ae = pd.DataFrame(columns=["mrn", "time", "event"])

    surv = sub_l1[["mrn", "censor_days"]].merge(first_ae, on="mrn", how="left")
    surv["event"] = surv["event"].fillna(0).astype(int)
    surv.loc[surv["event"] == 0, "time"] = surv.loc[surv["event"] == 0, "censor_days"]
    surv.loc[(surv["event"] == 1) & (surv["time"] > surv["censor_days"]), "event"] = 0
    surv.loc[surv["event"] == 0, "time"] = surv["censor_days"]
    surv = surv[surv["time"] > 0].copy()
    surv["time_months"] = surv["time"] / DAYS_PER_MONTH
    return surv[["mrn", "time_months", "event"]]


def compute_group_pvalues(group_order, mrn_sets, ae_list):
    """Two-group log-rank test p-value per toxicity, for the bracket
    annotation above each pair of bars."""
    g1, g2 = group_order[0], group_order[1]
    pvals = {}
    for tox in ae_list:
        s1 = build_surv_table(mrn_sets[g1], tox)
        s2 = build_surv_table(mrn_sets[g2], tox)
        if len(s1) < 2 or len(s2) < 2:
            pvals[tox] = np.nan
            continue
        result = logrank_test(s1["time_months"], s2["time_months"],
                               event_observed_A=s1["event"], event_observed_B=s2["event"])
        pvals[tox] = result.p_value
    return pvals

## 8. Compute cumulative incidence and p-values for both strata

In [ ]:
ae_list = TOXICITY_COLUMNS

print("Computing Stratum A (ICI vs Non-ICI)...")
stratum_a_results = {
    grp: [ci_at_t(stratum_a_mrns[grp], tox) for tox in ae_list]
    for grp in STRATUM_A_ORDER
}
stratum_a_pvals = compute_group_pvalues(STRATUM_A_ORDER, stratum_a_mrns, ae_list)

print("Computing Stratum B (CTLA-4 vs PD-(L)1)...")
stratum_b_results = {
    grp: [ci_at_t(stratum_b_mrns[grp], tox) for tox in ae_list]
    for grp in STRATUM_B_ORDER
}
stratum_b_pvals = compute_group_pvalues(STRATUM_B_ORDER, stratum_b_mrns, ae_list)

print("Done.")
print("\nStratum A log-rank p-values:", {TOXICITY_DISPLAY[t]: round(p, 4) for t, p in stratum_a_pvals.items()})
print("Stratum B log-rank p-values:", {TOXICITY_DISPLAY[t]: round(p, 4) for t, p in stratum_b_pvals.items()})

## 9. Plotting function

Including:
`rcParams` (Arial, `pdf.fonttype`/`ps.fonttype` 42, no grid, top/right spines
off, 450 dpi), `PLOT_HEIGHT_IN = 1.7`, `total_width = 0.95`, `ylabel`
fontsize 7, x-tick rotation 30°/`ha="right"`/fontsize 6, y-tick labelsize 6,
legend styling (fontsize 5, `framealpha=0.95`, `loc="upper center"`,
`bbox_to_anchor=(0.5, 1.18)`, `handlelength=1`, `columnspacing=0.8`), error
bar linewidth 0.5 / capsize 1.5 / no bar edge color, and exporting via
`bbox_inches="tight"`.

Three deliberate departures, all functionally necessary rather than stylistic
drift:

1. **Figure width** (4.5in vs. the source's 8in) — this figure has 2 bars per
   AE cluster instead of 9, so it needs less width; `PLOT_HEIGHT_IN` and all
   font sizes are unchanged.
2. **Y-axis headroom** (1.45× the tallest error bar vs. the source's 1.1×) —
   extra room is needed to fit the p-value bracket + label, a feature the
   source figure doesn't have.
3. **Bottom margin / export.** The source notebook already exports with
   `bbox_inches="tight"` (see its own save cell), which this notebook now
   matches — this is what fixes the label-clipping issue from before,
   regardless of label length or rotation.

In [ ]:
def set_axes_position_inches(fig, ax, left_in, top_in, width_in, height_in):
    fw, fh = fig.get_size_inches()
    ax.set_position([
        left_in / fw,
        1 - (top_in + height_in) / fh,
        width_in / fw,
        height_in / fh,
    ])


def format_pvalue(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    if p < 0.001:
        return "p < 0.001"
    return f"p = {p:.3f}"


def plot_two_group_ci(group_order, results, colors, group_n, p_values=None,
                       title=None, fig_width=4.5, save_path=None):
    """Grouped bar chart of 1-year cumulative incidence, two groups per AE,
    with a log-rank p-value bracket above each pair of bars."""
    TOP_MARGIN_IN = 0.9     
    PLOT_HEIGHT_IN = 1.7
    BOTTOM_MARGIN_IN = 0.45   
    LEFT_MARGIN_IN = 0.70
    RIGHT_MARGIN_IN = 0.30

    fig_height = TOP_MARGIN_IN + PLOT_HEIGHT_IN + BOTTOM_MARGIN_IN
    plot_width_in = fig_width - LEFT_MARGIN_IN - RIGHT_MARGIN_IN

    n_ae = len(ae_list)
    display_names = [TOXICITY_DISPLAY[t] for t in ae_list]
    n_groups = len(group_order)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    total_width = 0.95  
    bar_width = total_width / n_groups
    x = np.arange(n_ae)
    offset = -total_width / 2

    group_hi = {}
    group_x = {}
    for j, grp in enumerate(group_order):
        pct, lo, hi = zip(*results[grp])
        pct, lo, hi = np.array(pct), np.array(lo), np.array(hi)
        bar_x = x + offset + (j + 0.5) * bar_width
        ax.bar(bar_x, pct, bar_width,
               yerr=[pct - lo, hi - pct], capsize=1.5,
               color=colors.get(grp, "#888"), edgecolor="none",
               label=f"{grp} (N={group_n[grp]:,})",
               ecolor="gray", error_kw={"linewidth": 0.5})
        group_hi[grp] = hi
        group_x[grp] = bar_x

    # Headroom for the p-value brackets: scale further above the tallest
    # error bar than a plain bar chart would need, capped at 50%.
    base_max = max(np.max(group_hi[g]) for g in group_order)
    y_max = min(base_max * 1.45, 50) if base_max > 0 else 1.0

    if p_values is not None:
        for i, tox in enumerate(ae_list):
            g1, g2 = group_order[0], group_order[1]
            x1, x2 = group_x[g1][i], group_x[g2][i]
            local_top = max(group_hi[g1][i], group_hi[g2][i])
            bracket_top = local_top + y_max * 0.05
            tick = y_max * 0.015
            ax.plot([x1, x1, x2, x2],
                    [bracket_top - tick, bracket_top, bracket_top, bracket_top - tick],
                    color="black", linewidth=0.5, clip_on=False)
            ax.text((x1 + x2) / 2, bracket_top + y_max * 0.015,
                    format_pvalue(p_values.get(tox)),
                    ha="center", va="bottom", fontsize=5, clip_on=False)

    ax.xaxis.set_label_coords(0.5, -0.22)
    ax.set_ylabel("1-year cumulative" + chr(10) + "incidence (%, 95% CI)", fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(display_names, rotation=30, ha="right", rotation_mode="anchor", fontsize=6)
    ax.tick_params(axis="y", labelsize=6)
    ax.legend(fontsize=5, framealpha=0.95, ncol=2, loc="upper center",
              bbox_to_anchor=(0.5, 1.18), handlelength=1, columnspacing=0.8)
    ax.set_ylim(0, y_max)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if title:
        ax.set_title(title, fontsize=7, pad=18)

    set_axes_position_inches(fig, ax, left_in=LEFT_MARGIN_IN, top_in=TOP_MARGIN_IN,
                              width_in=plot_width_in, height_in=PLOT_HEIGHT_IN)

    if save_path:
        # bbox_inches="tight" crops/expands the exported page to whatever was
        # actually drawn (rotated tick labels, legend, p-value brackets), so
        # nothing gets clipped regardless of label length, rotation angle, or
        # font size — a fixed margin is fragile because the widest label
        # ("Adrenal Insufficiency") can run past a hardcoded left/bottom edge.
        fig.savefig(save_path, bbox_inches="tight", pad_inches=0.03)
        print(f"Saved: {save_path}")

    return fig, ax

## 10. Figure A — ICI vs Non-ICI

In [ ]:
group_n_a = {g: len(stratum_a_mrns[g]) for g in STRATUM_A_ORDER}

fig_a, ax_a = plot_two_group_ci(
    STRATUM_A_ORDER, stratum_a_results, STRATUM_A_COLORS, group_n_a,
    p_values=stratum_a_pvals,
    save_path=f"{RESULTS_DIR}/Cumulative_Incidence_ICI_vs_nonICI_2C.pdf",
)

## 11. Figure B — CTLA-4 (± PD-(L)1) vs PD-(L)1 (no CTLA-4), within ICI

In [ ]:
group_n_b = {g: len(stratum_b_mrns[g]) for g in STRATUM_B_ORDER}

fig_b, ax_b = plot_two_group_ci(
    STRATUM_B_ORDER, stratum_b_results, STRATUM_B_COLORS, group_n_b,
    p_values=stratum_b_pvals,
    save_path=f"{RESULTS_DIR}/Cumulative_Incidence_PD1_vs_CTLA4_2D.pdf",
)

## 11b. Build 1-year cumulative incidence table (both strata)

Same numbers plotted in Figures A and B above (KM-based 1-year cumulative
incidence, exponential-Greenwood/log-log 95% CI). AE rows are in the same
order as the figures (Figure 2b order): Liver Toxicity, Hypothyroidism,
Pneumonitis, Colitis, Adrenal Insufficiency, Hyperthyroidism.

**Note on p-values / team policy:** the two-group log-rank p-value (drawn as
the bracket annotation in Figures A/B) is an *unadjusted* comparison of the
raw KM curves. Per team policy, that unadjusted p-value must never be used
as "the" p-value in any CSV output for these Cox-modeled endpoints -- CSV
tables report **only** the adjusted, multivariable ridge-penalized Cox
p-value (the `exposure` row of `cox_results`, adjusted for concomitant
chemo/hormone/biologic/targeted therapy, cancer type, age, and sex).

Because of this, the table below is built here (its log-rank column only
feeds the plot brackets already drawn above), but the **CSV export itself
is deferred to Section 12b**, after the Cox models are fit, so the exported
file carries the Cox p-value instead of the log-rank one.


In [ ]:
def build_incidence_table(group_order, results, group_n, p_values, stratum_label):
    """Long-format table of 1-year cumulative incidence + 95% CI per
    group x toxicity, in the same AE order used by the figures, plus the
    log-rank p-value (same one drawn as the bracket in the plot).

    IMPORTANT: the `logrank_p_value` column produced here is for the plot
    brackets ONLY. Per team policy it must never be exported as "the"
    p-value for these Cox-modeled endpoints. The actual CSV export happens
    in Section 12b, after the Cox models are fit, where this column is
    replaced with the adjusted Cox-model p-value.
    """
    rows = []
    for tox_idx, tox in enumerate(ae_list):
        for grp in group_order:
            pct, lo, hi = results[grp][tox_idx]
            rows.append({
                "stratum": stratum_label,
                "group": grp,
                "n_group": group_n[grp],
                "toxicity": TOXICITY_DISPLAY[tox],
                "cum_incidence_1yr_pct": pct,
                "ci_95_lower": lo,
                "ci_95_upper": hi,
                "logrank_p_value": (p_values or {}).get(tox, np.nan),
            })
    df = pd.DataFrame(rows)
    df["toxicity"] = pd.Categorical(df["toxicity"],
                                     categories=[TOXICITY_DISPLAY[t] for t in ae_list],
                                     ordered=True)
    return df.sort_values(["toxicity", "group"]).reset_index(drop=True)


incidence_a = build_incidence_table(STRATUM_A_ORDER, stratum_a_results, group_n_a,
                                     stratum_a_pvals, "A. ICI vs non-ICI")
incidence_b = build_incidence_table(STRATUM_B_ORDER, stratum_b_results, group_n_b,
                                     stratum_b_pvals, "B. Within ICI")

# No to_csv() here on purpose -- see Section 12b for the actual export,
# which swaps in the adjusted Cox p-value before writing the CSV.
incidence_a

## 12. Cox proportional-hazards models

Patient-level model frame: Line-1 patients with known sex, top-7 cancer types
kept as indicators (everything else, including multi-cancer, folded into
`Other`, which is the reference level).

Uses `lifelines.CoxPHFitter` with a light ridge penalty (`penalizer=0.1`),
mirroring the R script's `ridge(theta = 0.1, scale = TRUE)`. As in the R
script, the exact shrinkage won't match numerically (the two implementations
normalize the penalized log-likelihood differently — sum vs. mean), but the
covariate-selection logic (drop zero-event-cell covariates, then drop
lowest-priority covariates until events-per-parameter ≥ 10) is replicated
exactly.

In [ ]:
model_df = patient_covars.merge(line1, on="mrn", how="inner")
model_df = model_df[model_df["sex"] != "Unknown"].copy()

model_df["contains_chemo"] = model_df.apply(lambda r: int(on(r, "contains_chemo")), axis=1)
model_df["contains_hormone"] = model_df.apply(lambda r: int(on(r, "contains_hormone")), axis=1)
model_df["contains_biologic"] = model_df.apply(lambda r: int(on(r, "contains_biologic")), axis=1)
model_df["contains_targeted"] = model_df.apply(lambda r: int(on(r, "contains_targeted")), axis=1)
model_df["age"] = model_df["age_at_lot_start"]
model_df["sex_male"] = (model_df["sex"] == "MALE").astype(int)

top7 = (model_df[model_df["cancer_type"] != "Multiple Cancer Type Patient"]
        ["cancer_type"].value_counts().head(7).index.tolist())

model_df["cancer_type_binned"] = np.where(model_df["cancer_type"].isin(top7),
                                           model_df["cancer_type"], "Other")
cancer_dummies = pd.get_dummies(model_df["cancer_type_binned"], prefix="cancer")
if "cancer_Other" in cancer_dummies.columns:
    cancer_dummies = cancer_dummies.drop(columns=["cancer_Other"])
cancer_dummies = cancer_dummies.astype(int)
model_df = pd.concat([model_df, cancer_dummies], axis=1)

cancer_dummy_cols = sorted(cancer_dummies.columns.tolist())
concomitant_cols = ["contains_chemo", "contains_hormone", "contains_biologic", "contains_targeted"]

print(f"Model cohort: {len(model_df):,} patients")
print(f"Top 7 cancer types (rest -> Other): {top7}")

In [ ]:
def fit_cox_model(cox_df, priority_covariates, binary_covariates):
    """Replicates the R script's zero-cell / EPV covariate-dropping logic,
    then fits a ridge-penalized Cox model with `exposure` as the first term."""
    n_total = len(cox_df)
    n_events = int(cox_df["event"].sum())
    n_exposed = int(cox_df["exposure"].sum())

    # Step 1: drop binary covariates with a zero-event cell (age is exempt).
    zero_cell = []
    for c in priority_covariates:
        if c not in binary_covariates:
            continue
        if cox_df[c].nunique() < 2:
            zero_cell.append(c)
            continue
        events_by_level = cox_df.groupby(c)["event"].sum()
        if (events_by_level == 0).any():
            zero_cell.append(c)
    covariates = [c for c in priority_covariates if c not in zero_cell]

    # Step 2: drop lowest-priority covariates until events-per-parameter >= 10.
    epv_dropped = []
    while len(covariates) > 0 and n_events / (len(covariates) + 1) < MIN_EVENTS_PER_PARAM:
        epv_dropped.append(covariates[-1])
        covariates = covariates[:-1]

    cols = ["exposure"] + covariates + ["time_months", "event"]
    fit_df = cox_df[cols].dropna()

    try:
        cph = CoxPHFitter(penalizer=PENALIZER, l1_ratio=0.0)
        cph.fit(fit_df, duration_col="time_months", event_col="event")
        summ = cph.summary
        out = pd.DataFrame({
            "covariate": summ.index,
            "hr": summ["exp(coef)"].values,
            "hr_lower_95": summ["exp(coef) lower 95%"].values,
            "hr_upper_95": summ["exp(coef) upper 95%"].values,
            "p": summ["p"].values,
            "model_status": "fit",
        })
    except Exception as e:
        out = pd.DataFrame({
            "covariate": [np.nan], "hr": [np.nan], "hr_lower_95": [np.nan],
            "hr_upper_95": [np.nan], "p": [np.nan],
            "model_status": [f"failed: {e}"],
        })

    out["n_total"] = n_total
    out["n_events"] = n_events
    out["n_exposed"] = n_exposed
    out["covariates_dropped_zero_cell"] = ", ".join(zero_cell)
    out["covariates_dropped_low_epv"] = ", ".join(epv_dropped)
    return out

In [ ]:
# `build_surv_table` is already defined in Section 7 and reused here for the
# Cox loop, so it isn't redefined below.

strata_specs = [
    dict(name="A. ICI vs non-ICI", slug="ici_vs_nonici",
         contrast="ICI vs non-ICI",
         cohort=model_df.copy(),
         exposure_col="ici_group", exposed_level="ICI",
         drop_covariates=[]),
    dict(name="B. Within ICI", slug="ctla4_vs_pd1",
         contrast="CTLA-4 (\u00b1 PD-(L)1) vs PD-(L)1 (no CTLA-4)",
         cohort=model_df[model_df["ici_agent"].isin(STRATUM_B_ORDER)].copy(),
         exposure_col="ici_agent", exposed_level=STRATUM_B_ORDER[0],
         drop_covariates=[]),
]

cox_rows = []

for spec in strata_specs:
    cohort = spec["cohort"].copy()
    cohort["exposure"] = (cohort[spec["exposure_col"]] == spec["exposed_level"]).astype(int)

    priority_covariates = [c for c in (concomitant_cols + cancer_dummy_cols + ["age", "sex_male"])
                            if c not in spec["drop_covariates"]]
    binary_covariates = [c for c in (concomitant_cols + cancer_dummy_cols + ["sex_male"])
                          if c not in spec["drop_covariates"]]

    for tox in TOXICITY_COLUMNS:
        surv_tox = build_surv_table(set(cohort["mrn"]), tox)
        cox_df = cohort.merge(surv_tox, on="mrn", how="inner")
        cox_df = cox_df.dropna(subset=["age", "sex_male", "time_months", "event"])

        out = fit_cox_model(cox_df, priority_covariates, binary_covariates)
        out["stratum"] = spec["name"]
        out["stratum_slug"] = spec["slug"]
        out["contrast"] = spec["contrast"]
        out["toxicity"] = TOXICITY_DISPLAY[tox]
        out["model_id"] = spec["slug"] + "__" + tox
        cox_rows.append(out)

        n_exp = int(cox_df["exposure"].sum())
        n_ev = int(cox_df["event"].sum())
        print(f"  {spec['name']} / {TOXICITY_DISPLAY[tox]}: "
              f"n={len(cox_df):,}, exposed={n_exp:,}, events={n_ev:,}")

cox_results = pd.concat(cox_rows, ignore_index=True)
cox_results = cox_results[["model_id", "stratum", "contrast", "toxicity", "covariate",
                            "hr", "hr_lower_95", "hr_upper_95", "p",
                            "n_total", "n_exposed", "n_events",
                            "covariates_dropped_zero_cell", "covariates_dropped_low_epv",
                            "model_status"]]

cox_results.head()

In [ ]:
# The exposure row is the headline result for each model.
cox_results[cox_results["covariate"] == "exposure"]

## 12b. Attach Cox p-values to the incidence tables and export CSVs

Per team policy, CSV exports for these Cox-modeled toxicity endpoints must
report the adjusted Cox-model p-value only -- never the unadjusted
log-rank p-value. This takes the incidence tables built in Section 11b,
drops their `logrank_p_value` column, joins in the exposure-row p-value
from `cox_results` (same model fit in Section 12) on `stratum` +
`toxicity`, and writes out the final incidence CSVs.


In [ ]:
# Exposure-row Cox p-value per stratum x toxicity -- the only p-value that
# should appear in any exported CSV for these endpoints.
cox_exposure_p = (
    cox_results[cox_results["covariate"] == "exposure"]
    [["stratum", "toxicity", "p"]]
    .rename(columns={"p": "cox_p_value"})
)

def attach_cox_p_and_export(incidence_df, out_path):
    """Replace the unadjusted log-rank p-value with the adjusted Cox
    exposure p-value, then write the CSV. Raises if any row can't be
    matched to a Cox result, rather than silently exporting a NaN."""
    df = incidence_df.drop(columns=["logrank_p_value"]).merge(
        cox_exposure_p, on=["stratum", "toxicity"], how="left"
    )
    missing = df["cox_p_value"].isna().sum()
    if missing:
        raise ValueError(
            f"{missing} row(s) in {out_path} have no matching Cox p-value -- "
            "check that stratum/toxicity labels line up with cox_results."
        )
    df.to_csv(out_path, index=False)
    return df

incidence_a_final = attach_cox_p_and_export(
    incidence_a, f"{RESULTS_DIR}/Cumulative_Incidence_ICI_vs_nonICI_2C_incidence.csv")
incidence_b_final = attach_cox_p_and_export(
    incidence_b, f"{RESULTS_DIR}/Cumulative_Incidence_PD1_vs_CTLA4_2E_incidence.csv")

print("Saved incidence CSVs with cox_p_value (adjusted Cox model, exposure row) to", RESULTS_DIR)
incidence_a_final

## 13. Export Cox Models

In [ ]:
cox_results[cox_results["stratum"] == "A. ICI vs non-ICI"].to_csv(
    f"{RESULTS_DIR}/Cumulative_Incidence_ICI_vs_nonICI_2C_cox_models.csv", index=False)
cox_results[cox_results["stratum"] == "B. Within ICI"].to_csv(
    f"{RESULTS_DIR}/Cumulative_Incidence_PD1_vs_CTLA4_2D_cox_models.csv", index=False)

print("Saved Cox model CSVs to", RESULTS_DIR)

# ============================================================================
# REGENERATE FIGURES WITH COX P-VALUES (not log-rank)
# The earlier figures (Sections 10-11) used unadjusted log-rank p-values.
# Per team policy, figures should display the adjusted Cox p-values.
# ============================================================================

# Extract Cox exposure p-values as dictionaries keyed by toxicity
cox_p_a = cox_results[(cox_results["stratum"] == "A. ICI vs non-ICI") & 
                       (cox_results["covariate"] == "exposure")].set_index("toxicity")["p"].to_dict()
cox_p_b = cox_results[(cox_results["stratum"] == "B. Within ICI") & 
                       (cox_results["covariate"] == "exposure")].set_index("toxicity")["p"].to_dict()

# Map display names back to internal names for the p-value lookup
tox_display_to_internal = {v: k for k, v in TOXICITY_DISPLAY.items()}
cox_pvals_a = {tox_display_to_internal[k]: v for k, v in cox_p_a.items()}
cox_pvals_b = {tox_display_to_internal[k]: v for k, v in cox_p_b.items()}

print("\nRegenerating figures with Cox p-values (adjusted)...")
print(f"  Stratum A Cox p-values: { {TOXICITY_DISPLAY[k]: f'{v:.4f}' for k, v in cox_pvals_a.items()} }")
print(f"  Stratum B Cox p-values: { {TOXICITY_DISPLAY[k]: f'{v:.4f}' for k, v in cox_pvals_b.items()} }")

# Regenerate Figure 2C with Cox p-values
fig_a_cox, _ = plot_two_group_ci(
    STRATUM_A_ORDER, stratum_a_results, STRATUM_A_COLORS, group_n_a,
    p_values=cox_pvals_a,
    save_path=f"{RESULTS_DIR}/Cumulative_Incidence_ICI_vs_nonICI_2C.pdf",
)
plt.close(fig_a_cox)

# Regenerate Figure 2D with Cox p-values
fig_b_cox, _ = plot_two_group_ci(
    STRATUM_B_ORDER, stratum_b_results, STRATUM_B_COLORS, group_n_b,
    p_values=cox_pvals_b,
    save_path=f"{RESULTS_DIR}/Cumulative_Incidence_PD1_vs_CTLA4_2D.pdf",
)
plt.close(fig_b_cox)

print("Regenerated figures 2C and 2D with adjusted Cox p-values.")

## 14. Sanity check — confirm CSV p-values match the Cox models

Run this after Sections 11b, 12b, and 13 (i.e. after all CSVs have been
(re)written) to confirm every `*_incidence.csv` reports the same p-value,
per stratum x toxicity, as the `exposure` row of the matching
`*_cox_models.csv` -- i.e. that no unadjusted log-rank p-value snuck back
into an exported file.


In [ ]:
# Re-read the exported CSVs from disk and confirm the p-value reported in
# each *_incidence.csv exactly matches the exposure-row p-value in the
# corresponding *_cox_models.csv, for every stratum x toxicity combination.

check_pairs = [
    ("A. ICI vs non-ICI",
     f"{RESULTS_DIR}/Cumulative_Incidence_ICI_vs_nonICI_2C_incidence.csv",
     f"{RESULTS_DIR}/Cumulative_Incidence_ICI_vs_nonICI_2C_cox_models.csv"),
    ("B. Within ICI",
     f"{RESULTS_DIR}/Cumulative_Incidence_PD1_vs_CTLA4_2D_incidence.csv",
     f"{RESULTS_DIR}/Cumulative_Incidence_PD1_vs_CTLA4_2D_cox_models.csv"),
]

all_ok = True
for stratum_name, incidence_path, cox_path in check_pairs:
    inc = pd.read_csv(incidence_path)
    cox = pd.read_csv(cox_path)

    if "cox_p_value" not in inc.columns:
        raise AssertionError(
            f"{incidence_path} has no 'cox_p_value' column -- it still has "
            "the old unadjusted log-rank p-value. Re-run Sections 11b/12b."
        )
    if "logrank_p_value" in inc.columns:
        raise AssertionError(
            f"{incidence_path} still has a 'logrank_p_value' column -- "
            "an unadjusted p-value is being exported. Re-run Sections 11b/12b."
        )

    cox_exp = cox[cox["covariate"] == "exposure"][["toxicity", "p"]].rename(
        columns={"p": "cox_p_value_ref"})
    merged = (inc[["toxicity", "cox_p_value"]].drop_duplicates()
              .merge(cox_exp, on="toxicity", how="outer", indicator=True))

    mismatched = merged[
        (merged["_merge"] != "both") |
        ~np.isclose(merged["cox_p_value"], merged["cox_p_value_ref"], equal_nan=True)
    ]

    if len(mismatched):
        all_ok = False
        print(f"[MISMATCH] {stratum_name}:")
        print(mismatched)
    else:
        print(f"[OK] {stratum_name}: incidence CSV p-values match cox_models CSV "
              f"exposure p-values for all {len(merged)} toxicities.")

if all_ok:
    print("\nAll incidence CSVs are consistent with the Cox model CSVs -- "
          "the only p-value being reported anywhere is the adjusted Cox p-value.")
else:
    raise AssertionError("p-value mismatch between incidence and cox_models CSVs -- see above.")